# Scrape antifakenewscenter.com – #มิจฉาชีพ news

Keep only posts tagged **#มิจฉาชีพ** together with **#อาชญากรรมออนไลน์** and/or **#ข่าวปลอม**.

The site is WordPress, so instead of parsing the tag page HTML we read the same posts of the
`/tag/มิจฉาชีพ/` archive through its public REST API (`/wp-json/wp/v2/posts?tags=<id>`). Each post
already carries its tag IDs, so no per-post page fetch is needed.

Output columns: `post_id` (the site's WordPress post ID), `date` (publish date, `DD/MM/YYYY` in the
CSV and a datetime column in the dataframe), `hashtag`, `title`, `content`, `Channel`, `ORG`, `scam_type` and the three `method_` columns.

`Channel` and `ORG` are labelled with keyword rules on the title and content, so they are a first
pass, not ground truth:
- `Channel`: `call` if the scammer phones the victim, otherwise `text`.
- `ORG`: the Thai organization(s) the scam is about, from the reference list built further
  down (see "Organization column"). Empty when no listed organization is found.
- `scam_type`: one of the five Callas scam types, labelled by Qwen (see "Scam type and method columns").
- `method_channel`, `method_scammer_action`, `method_victim_action`: how the scam works, in
  short phrases, labelled by the same Qwen call (see "Scam method columns").

In [11]:
import html
import io
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

API = "https://www.antifakenewscenter.com/wp-json/wp/v2"
HEADERS = {"User-Agent": "Mozilla/5.0 (research scraper; Callas project)"}
DELAY = 1.0  # seconds between requests

BASE_TAG = "มิจฉาชีพ"
TARGET_TAGS = ["อาชญากรรมออนไลน์", "ข่าวปลอม"]
OUT_PATH = Path("data/antifakenews_scam.csv")

In [12]:
def get_json(path, params, retries=3):
    """GET an API endpoint with retries; returns (json, response headers)."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{API}/{path}", params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json(), r.headers
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 * attempt)


def find_tag_id(name):
    """Resolve a tag name (with or without a leading '#') to its WordPress tag ID."""
    tags, _ = get_json("tags", {"search": name, "per_page": 100, "_fields": "id,name,count"})
    exact = [t for t in tags if t["name"].lstrip("#").strip() == name]
    if not exact:
        raise ValueError(f"tag not found: {name}")
    return max(exact, key=lambda t: t["count"])["id"]


tag_ids = {name: find_tag_id(name) for name in [BASE_TAG, *TARGET_TAGS]}
tag_ids

{'มิจฉาชีพ': 88, 'อาชญากรรมออนไลน์': 5713, 'ข่าวปลอม': 11}

In [13]:
def fetch_posts(tag_id):
    """All posts of a tag (same set as the /tag/<slug>/ archive pages)."""
    posts, page, total_pages = [], 1, 1
    while page <= total_pages:
        batch, headers = get_json("posts", {"tags": tag_id, "per_page": 100, "page": page,
                                            "_fields": "id,date,title,content,tags"})
        total_pages = int(headers["X-WP-TotalPages"])
        posts += batch
        print(f"page {page}/{total_pages}: {len(posts)}/{headers['X-WP-Total']} posts")
        page += 1
        time.sleep(DELAY)
    return posts


raw_posts = fetch_posts(tag_ids[BASE_TAG])

page 1/9: 100/863 posts
page 2/9: 200/863 posts
page 3/9: 300/863 posts
page 4/9: 400/863 posts
page 5/9: 500/863 posts
page 6/9: 600/863 posts
page 7/9: 700/863 posts
page 8/9: 800/863 posts
page 9/9: 863/863 posts


In [14]:
FOOTER_HEADING = "ช่องทางการติดตามและแจ้งเบาะแสข่าวปลอม"
FOOTER_START = "LINE : @antifakenewscenter"  # the site's contact block closes every post from here on

# The scammer phones the victim -> "call", anything else -> "text".
# Only phrases about being called count; a bare "โทรศัพท์" or a report hotline does not.
CALL = (r"คอลเซ็นเตอร์|ทางโทรศัพท์|ผ่านโทรศัพท์|(?i:\bcall\s*center\b)|รับสาย|ได้รับสาย"
        r"|โทรศัพท์(?:มา|หา|ติดต่อ|สอบ|แจ้ง|ถาม|ติดตาม)"
        r"|โทร(?:มา|หา|ไป|แจ้ง|ติดต่อ|สอบ|ถาม|ตรวจสอบ|หลอก|ขู่|เข้า)")
HOTLINE = r"(?i:call\s*center)\s*\d{3,4}|สายด่วน\s*\d+"  # e.g. "Call Center 1284", not a scam call


def clean_html(fragment):
    soup = BeautifulSoup(fragment, "html.parser")
    for tag in soup(["script", "style", "iframe", "figure", "img"]):
        tag.decompose()
    lines = (re.sub(r"[ \t]+", " ", line).strip() for line in soup.get_text("\n").replace("\xa0", " ").splitlines())
    lines = [line for line in lines if line]
    # drop the contact footer (LINE, website, Facebook, X, TikTok, IG) and its heading
    for i, line in enumerate(lines):
        if line.startswith(FOOTER_START):
            lines = lines[:i]
            if lines and lines[-1] == FOOTER_HEADING:
                lines.pop()
            break
    return "\n".join(lines)


def label_channel(text):
    return "call" if re.search(CALL, re.sub(HOTLINE, "", text)) else "text"


rows = []
for p in raw_posts:  # all fetched under #มิจฉาชีพ; keep those that also carry a target hashtag
    matched = [name for name in TARGET_TAGS if tag_ids[name] in p["tags"]]
    if matched:
        rows.append({
            "post_id": p["id"],
            "date": p["date"],
            "hashtag": ", ".join(matched),  # both tags -> one row, e.g. "อาชญากรรมออนไลน์, ข่าวปลอม"
            "title": html.unescape(BeautifulSoup(p["title"]["rendered"], "html.parser").get_text()).strip(),
            "content": clean_html(p["content"]["rendered"]),
        })

df = pd.DataFrame(rows)
# real datetime64 column, so it sorts chronologically (a DD/MM/YYYY string would sort by day first)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df["Channel"] = (df["title"] + "\n" + df["content"]).map(label_channel)
df = df.sort_values("date", ascending=False, ignore_index=True)
df

,post_id,date,hashtag,title,content,Channel
0,93951,2026-10-25,ข่าวปลอม,การไฟฟ้านครหลวง เปิดให้บริการผ่าน Line ID 1130270,เตือนภัย!! การไฟฟ้านครหลวง (กฟน. หรือ MEA) เปิ...,text
1,93944,2026-10-25,อาชญากรรมออนไลน์,ตรวจสอบให้ชัวร์ ใบแจ้งค่าเทอมปลอม,ช่วงใกล้เปิดเทอม ผู้ปกครองและนักเรียน–นักศึกษา...,text
2,93919,2026-10-24,ข่าวปลอม,ตำรวจเปิดเพจเฟซบุ๊ก ศูนย์ให้คำปรึกษาและประสานง...,เพจปลอมมาอีกแล้ว!! โดยสวมรอยเป็นเพจหน่วยงานราช...,text
3,93911,2026-10-24,ข่าวปลอม,กรมการปกครอง เปิด TikTok ใหม่ โดยใช้ชื่อบัญชี ...,จากการตรวจสอบ พบผู้ไม่หวังดีเปิดบัญชี TikTok ใ...,text
4,93905,2026-10-24,อาชญากรรมออนไลน์,ทุนการศึกษาปลอม หลอกเก็บค่าสมัคร,เหล่านักเรียน นักศึกษา ที่กำลังมองหาทุนเพื่อแบ...,text
...,...,...,...,...,...,...
372,10371,2021-05-21,ข่าวปลอม,ข่าวปลอม อย่าแชร์! ธนาคารเอกชน เปิดเว็บไซต์ลงท...,ตามที่มีการเผยแพร่ข้อมูลผ่านโซเชียลมีเดีย เกี่...,text
373,9793,2021-04-23,ข่าวปลอม,ข่าวปลอม อย่าแชร์! ธ. กรุงเทพส่ง SMS แจ้งได้รั...,ตามที่มีการแชร์ภาพปรากฏในสื่อต่างๆ เกี่ยวกับปร...,text
374,9256,2021-03-24,ข่าวปลอม,ข่าวปลอม อย่าแชร์! กองสลากฯ ส่งหนังสือโครงการท...,ตามที่ได้มีการเผยแพร่เอกสารในสื่อต่างๆ เกี่ยวก...,text
375,7012,2020-11-20,ข่าวปลอม,ข่าวปลอม อย่าแชร์! การคลังให้เจ้าหน้าที่ติดต่อ...,ตามที่มีการเผยแพร่ในสื่อต่าง ๆ เกี่ยวกับประเด็...,call


In [15]:
# drop posts with no title or no content (e.g. image-only posts): there is nothing to label in them
empty = (df["title"].str.strip() == "") | (df["content"].str.strip() == "")
if empty.any():
    print(f"dropping {empty.sum()} posts with no title or content: {df.loc[empty, 'post_id'].tolist()}")
df = df[~empty].reset_index(drop=True)

# check the scrape now, before ORG and Qwen labelling, so a bad scrape stops here and costs no API calls
assert df["post_id"].is_unique
assert df["date"].notna().all()
assert df["hashtag"].str.contains("|".join(TARGET_TAGS)).all()
assert not df["content"].str.contains(FOOTER_START, regex=False).any()
print(f"{len(df)} posts passed the checks")

377 posts passed the checks


## Organization column

The `ORG` column names the Thai organization each post says the scammer pretended to be,
taken from a reference list of organizations with their **full names and abbreviations**:

| Source | What it gives |
|---|---|
| [DGA open data, "รายชื่อหน่วยงานของรัฐส่วนกลางระดับกรมหรือเทียบเท่า"](https://catalog-dga.data.go.th/dataset/di-open1-01) | 413 central agencies (departments, state enterprises, independent agencies) with Thai name, English name and Thai abbreviation. Only a few rows carry an English abbreviation. Each ministry is added too, as its own row. |
| [Thai Wikipedia, รายชื่อธนาคารในประเทศไทย](https://th.wikipedia.org/wiki/รายชื่อธนาคารในประเทศไทย) | Banks currently operating, with the SET ticker or the state-bank abbreviation (BBL, KTB, SCB, GSB ...). |
| [Wikidata](https://www.wikidata.org) + English Wikipedia | English acronyms. The Thai name is looked up on Wikidata, which links to the English article, whose first sentence usually gives the acronym, e.g. "The Royal Thai Police (RTP) ...". An acronym is kept only if the article is about Thailand and the acronym fits the English name, because the bracket sometimes holds the parent ministry instead. |

The merged list is saved to `data/thai_organizations.csv`. Most organizations still have no English
abbreviation (80 of 451 do), because many have no English Wikipedia article or the article gives none.

A post is matched on any alias of an
organization (full name, abbreviation, `ธ.`/`แบงก์` + bank name, English name) after removing spaces,
because Thai spacing is irregular. The title is searched first. If it names no organization, the most
mentioned one in the content is used, ignoring the police, since almost every post tells the reader to
report to them. Limits: private companies (HomePro, Lotus's, Singha ...) and local bodies are not in
the list, so those posts get an empty `ORG`, and a match means the post *mentions* the
organization, not strictly that the scammer impersonated it.

In [16]:
DGA_CSV = ("https://catalog-dga.data.go.th/dataset/af5f5723-0a54-47be-900f-19420585cc56/resource/"
           "64052093-d954-44f4-9993-d660f9fa245a/download/.csv")
BANKS_URL = "https://th.wikipedia.org/wiki/รายชื่อธนาคารในประเทศไทย"
ORG_PATH = Path("data/thai_organizations.csv")

has_thai = lambda s: bool(re.search(r"[฀-๿]", s))


def download(url, timeout=30):
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return r


def load_dga_orgs():
    """Agencies, plus each ministry (the list only has its permanent-secretary office)."""
    d = pd.read_csv(io.BytesIO(download(DGA_CSV, timeout=60).content), encoding="utf-8-sig")
    # OrgNameAbbr mixes Thai and English abbreviations, e.g. "BOI, สกท."
    abbrs = d["OrgNameAbbr"].fillna("").map(lambda s: [a.strip() for a in s.split(",") if a.strip()])
    agencies = pd.DataFrame({
        "name_th": d["OrgNameThai"].str.strip(),
        "name_en": d["OrgNameEnglish"].astype(str).str.strip(),
        "abbr_th": abbrs.map(lambda a: "|".join(x for x in a if has_thai(x))),
        "abbr_en": abbrs.map(lambda a: "|".join(x for x in a if not has_thai(x))),
        "type": d["MajorCategoryName_byLaw"],
        "source": "DGA open data",
    })
    offices = d[d["OrgNameThai"].str.startswith("สำนักงานปลัดกระทรวง")].drop_duplicates("MinistryNameThai")
    office_abbr = dict(zip(offices["MinistryNameThai"], offices["OrgNameAbbr"].astype(str).str.strip()))
    ministries = pd.DataFrame([
        {"name_th": m.strip(), "name_en": "", "abbr_en": "", "type": "กระทรวง", "source": "DGA open data (ministry)",
         # สป.กค. (office of the ministry) -> กค. (the ministry)
         "abbr_th": office_abbr[m][3:].lstrip() if office_abbr.get(m, "").startswith("สป.") else ""}
        for m in sorted(d["MinistryNameThai"].dropna().unique()) if m.startswith("กระทรวง")
    ])
    return pd.concat([agencies, ministries])


def load_bank_orgs():
    rows = []
    soup = BeautifulSoup(download(BANKS_URL).text, "html.parser")
    for table in soup.select("table.wikitable")[:5]:  # banks still operating
        for tr in table.find_all("tr")[1:]:
            c = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
            if len(c) >= 5 and c[1].startswith("ธนาคาร"):
                m = re.search(r"(?:SET\s*:|ตัวอักษรย่อ\s*:)\s*([A-Z]+)", c[3])  # "SET: BBL" / "ตัวอักษรย่อ: GSB"
                rows.append({"name_th": c[1], "name_en": "", "abbr_th": "", "abbr_en": m.group(1) if m else "",
                             "type": "ธนาคาร", "source": "Thai Wikipedia"})
    return pd.DataFrame(rows)


# --- English acronyms: Thai name -> Wikidata item -> English Wikipedia article -> "(RTP)" in its first sentence
WIKI_HEADERS = {"User-Agent": "CallasResearch/0.1 (https://github.com/ZenithGeeks/Callas; scam dataset)"}
ACRONYM_IN_INTRO = re.compile(r"\((?:abbreviated(?: as)?:?\s*|abbr\.?:?\s*)?([A-Z][A-Za-z]*[A-Z]|[A-Z](?:\.[A-Z])+\.?)\)"
                              r"|abbreviated(?: as)? ([A-Z]{2,})")


def wiki_get(url, **params):
    """Wikimedia APIs ask for a descriptive User-Agent and sometimes rate limit, so retry."""
    for attempt in range(5):
        r = requests.get(url, params={**params, "format": "json"}, headers=WIKI_HEADERS, timeout=60)
        if r.ok and r.headers.get("content-type", "").startswith("application/json"):
            return r.json()
        time.sleep(3 * (attempt + 1))
    r.raise_for_status()
    raise RuntimeError(f"no JSON from {url}")


def acronym_fits(acronym, name):
    """RTP fits "Royal Thai Police": same first letter, letters in order, last letter starts a word."""
    words = re.sub(r"\([^)]*\)", "", name).split()
    words = words[1:] if words and words[0].lower() == "the" else words
    letters, text = re.sub(r"[^A-Za-z]", "", acronym).upper(), " ".join(words).upper()
    if not words or not letters or letters[0] != text[0] or letters[-1] not in {w[0].upper() for w in words}:
        return False
    rest = iter(text)
    return all(ch in rest for ch in letters)


def load_wikipedia_acronyms(orgs):
    """English acronyms of the organizations whose English Wikipedia article gives one."""
    item_names = {}  # Wikidata item -> Thai names with that label
    for i in range(0, len(orgs), 100):
        values = " ".join(f'"{n}"@th' for n in orgs["name_th"].iloc[i:i + 100])
        r = requests.post("https://query.wikidata.org/sparql", headers={**WIKI_HEADERS, "Accept": "application/json"},
                          data={"query": f"SELECT ?item ?th WHERE {{ VALUES ?th {{ {values} }} ?item rdfs:label ?th }}"},
                          timeout=120)
        r.raise_for_status()
        for b in r.json()["results"]["bindings"]:
            item_names.setdefault(b["item"]["value"].rsplit("/", 1)[-1], []).append(b["th"]["value"])

    items, article = list(item_names), {}  # Wikidata item -> English article title
    for i in range(0, len(items), 50):
        entities = wiki_get("https://www.wikidata.org/w/api.php", action="wbgetentities", ids="|".join(items[i:i + 50]),
                            props="sitelinks", sitefilter="enwiki")["entities"]
        article |= {q: e["sitelinks"]["enwiki"]["title"] for q, e in entities.items() if "enwiki" in e.get("sitelinks", {})}

    titles, intro = sorted(set(article.values())), {}  # article title -> plain-text intro
    for i in range(0, len(titles), 20):
        q = wiki_get("https://en.wikipedia.org/w/api.php", action="query", prop="extracts", exintro=1, explaintext=1,
                     exlimit=20, redirects=1, titles="|".join(titles[i:i + 20]))["query"]
        renamed = {x["from"]: x["to"] for x in q.get("normalized", []) + q.get("redirects", [])}
        text = {p["title"]: p.get("extract", "") for p in q["pages"].values()}
        for t in titles[i:i + 20]:
            intro[t] = text.get(renamed.get(renamed.get(t, t), renamed.get(t, t)), "")

    name_en = dict(zip(orgs["name_th"], orgs["name_en"]))
    rows = []
    for item, title in article.items():
        m = ACRONYM_IN_INTRO.search(intro[title][:350])
        if not m or "Thai" not in intro[title][:600]:  # no acronym, or an article about another country
            continue
        acronym = m.group(1) or m.group(2)
        for th in item_names[item]:
            # the bracket sometimes holds the parent ministry (e.g. MoNRE), so the acronym must fit the name
            if any(acronym_fits(acronym, n) for n in (name_en.get(th, ""), title) if n):
                rows.append({"name_th": th, "abbr_en": acronym, "source": "English Wikipedia"})
    return pd.DataFrame(rows)


first_filled = lambda s: next((v for v in s if v), "")
join_unique = lambda s: "|".join(dict.fromkeys(a for v in s for a in v.split("|") if a))


def merge_orgs(frames):
    """One row per Thai name; abbreviations from every source are combined."""
    return (pd.concat(frames, ignore_index=True).fillna("")
            .groupby("name_th", sort=False, as_index=False)
            .agg(name_en=("name_en", first_filled), abbr_th=("abbr_th", join_unique),
                 abbr_en=("abbr_en", join_unique), type=("type", "first"), source=("source", "first")))


orgs = merge_orgs([load_dga_orgs(), load_bank_orgs()])  # state banks are in both sources
orgs = merge_orgs([orgs, load_wikipedia_acronyms(orgs)])
ORG_PATH.parent.mkdir(parents=True, exist_ok=True)
orgs.to_csv(ORG_PATH, index=False, encoding="utf-8-sig")
print(f"{len(orgs)} organizations -> {ORG_PATH}; {(orgs['abbr_en'] != '').sum()} with an English abbreviation")
orgs["type"].value_counts()

451 organizations -> data\thai_organizations.csv; 80 with an English abbreviation


type
ส่วนราชการ                                         163
สถาบันอุดมศึกษาของรัฐ                               87
รัฐวิสาหกิจ                                         52
องค์การมหาชนตามพระราชกฤษฎีกา                        37
หน่วยงานในกำกับหรือองค์การมหาชนตามพระราชบัญญัติ     31
กระทรวง                                             19
ธนาคาร                                              19
สภาวิชาชีพ                                          16
หน่วยงานอิสระของรัฐ                                 12
องค์กรอิสระตามรัฐธรรมนูญและองค์กรอัยการ              6
หน่วยงานอื่นของรัฐ                                   3
หน่วยงานของรัฐสภา                                    3
หน่วยงานของศาล                                       3
Name: count, dtype: int64

In [17]:
def squash(text):
    """Drop Thai spacing (it is irregular) but keep the gap between two Latin words."""
    text = text.replace("​", "")

    def gap(m):
        before = text[m.start() - 1] if m.start() else ""
        after = text[m.end()] if m.end() < len(text) else ""
        return " " if re.match(r"[A-Za-z0-9]", before) and re.match(r"[A-Za-z0-9]", after) else ""

    return re.sub(r"\s+", gap, text)


# colloquial names the lists do not carry: canonical Thai name -> extra aliases
MANUAL_ALIASES = {
    "สำนักงานตำรวจแห่งชาติ": ["ตำรวจ", "ตร."],
    "กรมสอบสวนคดีพิเศษ": ["ดีเอสไอ"],
    "ธนาคารแห่งประเทศไทย": ["แบงก์ชาติ"],
    "ธนาคารทหารไทยธนชาต": ["ทีทีบี"],
}
# ordinary words that must not stand in for an organization on their own
AMBIGUOUS_BANK_BARE = {"กรุงเทพ", "ไทย", "เครดิต"}  # keep "ธนาคาร"/"ธ." in front
GENERIC_COMPANY_BASE = {"ขนส่ง", "การบิน"}          # "บริษัท ขนส่ง จำกัด" minus บริษัท/จำกัด
BLOCKED_ALIASES = {"SME"}                           # loan product word, not the bank
IGNORE_IN_CONTENT = {"สำนักงานตำรวจแห่งชาติ"}        # posts end with "report to the police"


def org_aliases(row):
    name = row.name_th
    base = re.sub(r"\s*\((?:มหาชน|องค์การมหาชน)\)|\s*จำกัด|^บริษัท\s*", "", name).strip()
    out = {name, *MANUAL_ALIASES.get(name, []), *filter(None, row.abbr_en.split("|"))}
    if len(squash(base)) >= 6 and base not in GENERIC_COMPANY_BASE:
        out.add(base)
    if base.startswith("ธนาคาร"):
        rest = base[len("ธนาคาร"):].strip()
        out |= {"ธ." + rest, "แบงก์" + rest}
        if len(rest) >= 4 and rest not in AMBIGUOUS_BANK_BARE:
            out.add(rest)
    for a in filter(None, row.abbr_th.split("|")):
        out |= {a, re.sub(r"^สำนักงาน\s*", "", a)}  # "สำนักงาน ก.ล.ต." is also written "ก.ล.ต."
    # Thai abbreviations need >= 4 characters incl. dots, English ones >= 3, or they hit ordinary words
    return {a for a in out if a not in BLOCKED_ALIASES and len(squash(a)) >= (4 if has_thai(a) else 3)}


ALIAS_TO_ORG = {}
for row in orgs.itertuples():
    for alias in org_aliases(row):
        ALIAS_TO_ORG.setdefault(squash(alias), row.name_th)  # on a clash the first listed organization wins

# one pattern, longest alias first so the longest name wins; English aliases must be whole words
ALIAS_RE = re.compile("|".join(
    re.escape(a) if has_thai(a) else rf"(?<![A-Za-z]){re.escape(a)}(?![A-Za-z])"
    for a in sorted(ALIAS_TO_ORG, key=len, reverse=True)
))


def org_mentions(text):
    """Organizations named in the text, in order of appearance."""
    return [ALIAS_TO_ORG[m.group()] for m in ALIAS_RE.finditer(squash(text))]


def label_organization(title, content):
    in_title = dict.fromkeys(org_mentions(title))  # unique, in order
    if in_title:
        return ", ".join(in_title)
    counts = Counter(org for org in org_mentions(content) if org not in IGNORE_IN_CONTENT)
    return counts.most_common(1)[0][0] if counts else ""


df["ORG"] = [label_organization(t, c) for t, c in zip(df["title"], df["content"])]
labelled = df.loc[df["ORG"] != "", "ORG"]
print(f"{len(labelled)}/{len(df)} posts have an organization")
labelled.str.split(", ").explode().value_counts().head(20)

296/377 posts have an organization


ORG
สำนักงานป้องกันและปราบปรามการฟอกเงิน                     36
กรมการขนส่งทางบก                                         34
ธนาคารออมสิน                                             30
การไฟฟ้าส่วนภูมิภาค                                      22
สำนักงานคณะกรรมการกำกับหลักทรัพย์และตลาดหลักทรัพย์       22
ธนาคารกรุงไทย                                            16
ตลาดหลักทรัพย์แห่งประเทศไทย                              12
กรมพัฒนาธุรกิจการค้า                                      9
กรมการจัดหางาน                                            8
กระทรวงยุติธรรม                                           8
บริษัท ไปรษณีย์ไทย จำกัด                                  7
สำนักงานตำรวจแห่งชาติ                                     6
กระทรวงแรงงาน                                             5
กระทรวงการคลัง                                            5
ธนาคารแห่งประเทศไทย                                       5
สำนักงานประกันสังคม                                       5
กรมสรรพากร                          

## Scam type and method columns (labelled by Qwen)

Four columns are filled by Qwen, called through the university's OpenAI-compatible API
(`llm_labeler.py`):

- `scam_type`: one of the five Callas scam types (`fake_authority`, `romance`, `investment`,
  `phishing_otp`, `lottery`), or empty when the post describes no scam (an overview of several types,
  a fact-check of a rumour).
- `method_channel`, `method_scammer_action`, `method_victim_action`: how the scam works, in short Thai
  phrases (see "Scam method columns" below).

The model gets the whole of `scam_type_definitions.md` 

In [18]:
from llm_labeler import FIELDS, label_all

# 50 posts per API call, one call after another; returns the four label columns + llm_error
# (a run that stops early resumes from data/llm_labels_checkpoint.jsonl)
labels = label_all(df)
df[FIELDS] = labels[FIELDS]

print(f"{labels['llm_error'].sum()} posts failed to label")
df["scam_type"].replace("", "(none)").value_counts()

labelled 50/377
labelled 100/377
labelled 150/377
labelled 200/377
labelled 250/377
labelled 300/377
labelled 350/377
labelled 377/377
0 posts failed to label


scam_type
fake_authority    273
investment         40
phishing_otp       24
(none)             20
lottery            17
romance             3
Name: count, dtype: int64

## Scam method columns

Three short Thai fields say *how* each scam works, so the long `content` does not have to be read:

| Column | What it holds |
|---|---|
| `method_channel` | Where it happens: เพจเฟซบุ๊ก, กลุ่มเฟซบุ๊ก, TikTok, LINE, SMS, โทรศัพท์, เว็บไซต์, อีเมล, เอกสาร ... (comma separated) |
| `method_scammer_action` | The story or offer used to hook the victim |
| `method_victim_action` | What the victim is pushed to do |

They come from the same Qwen call as `scam_type` (cell above). The model is told to use the post's
own words and to leave a field empty when the post does not state it; channels outside the fixed
list are dropped. Posts with no `scam_type` have all three empty.

In [19]:
METHOD_COLUMNS = ["method_channel", "method_scammer_action", "method_victim_action"]
df[["title", *METHOD_COLUMNS]].head()

,title,method_channel,method_scammer_action,method_victim_action
0,การไฟฟ้านครหลวง เปิดให้บริการผ่าน Line ID 1130270,LINE,แอบอ้างเป็นบริการการไฟฟ้านครหลวงผ่าน LINE ID,แอดไลน์เพื่อติดต่อสอบถามหรือรับบริการ
1,ตรวจสอบให้ชัวร์ ใบแจ้งค่าเทอมปลอม,"SMS, QR Code",ส่ง SMS ปลอมแนบลิงก์ใบแจ้งค่าเทอมในนามสถานศึกษา,สแกน QR Code หรือกดลิงก์เพื่อชำระเงิน
2,ตำรวจเปิดเพจเฟซบุ๊ก ศูนย์ให้คำปรึกษาและประสานง...,เพจเฟซบุ๊ก,เปิดเพจปลอมอ้างเป็นศูนย์ให้คำปรึกษาและติดตามคว...,กดลิงก์หรือเปิดเผยข้อมูลส่วนตัว
3,กรมการปกครอง เปิด TikTok ใหม่ โดยใช้ชื่อบัญชี ...,TikTok,เปิดบัญชี TikTok ปลอมอ้างเป็นกรมการปกครอง,ส่งข้อมูลส่วนตัวหรือทำธุรกรรม
4,ทุนการศึกษาปลอม หลอกเก็บค่าสมัคร,เพจเฟซบุ๊ก,สร้างเพจรับสมัครทุนการศึกษาปลอมเรียกเก็บค่าประกัน,กรอกใบสมัครและส่งเอกสารส่วนตัว


In [20]:
print(f"{len(raw_posts)} posts under #{BASE_TAG} -> {len(df)} kept")
display(df["hashtag"].value_counts())
df["Channel"].value_counts()

863 posts under #มิจฉาชีพ -> 377 kept


hashtag
ข่าวปลอม                      321
อาชญากรรมออนไลน์               53
อาชญากรรมออนไลน์, ข่าวปลอม      3
Name: count, dtype: int64

Channel
text    330
call     47
Name: count, dtype: int64

In [21]:
# the data was checked right after scraping (see the cell after the scrape)
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# the date column is written as DD/MM/YYYY
df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig", date_format="%d/%m/%Y")

# read it back with the same format so the column is datetime again and sortable
# keep_default_na=False so empty labels show as "" instead of NaN
pd.read_csv(OUT_PATH, parse_dates=["date"], date_format="%d/%m/%Y", keep_default_na=False).head()

,post_id,date,hashtag,title,content,Channel,ORG,scam_type,method_channel,method_scammer_action,method_victim_action
0,93951,2026-10-25,ข่าวปลอม,การไฟฟ้านครหลวง เปิดให้บริการผ่าน Line ID 1130270,เตือนภัย!! การไฟฟ้านครหลวง (กฟน. หรือ MEA) เปิ...,text,การไฟฟ้านครหลวง,fake_authority,LINE,แอบอ้างเป็นบริการการไฟฟ้านครหลวงผ่าน LINE ID,แอดไลน์เพื่อติดต่อสอบถามหรือรับบริการ
1,93944,2026-10-25,อาชญากรรมออนไลน์,ตรวจสอบให้ชัวร์ ใบแจ้งค่าเทอมปลอม,ช่วงใกล้เปิดเทอม ผู้ปกครองและนักเรียน–นักศึกษา...,text,,fake_authority,"SMS, QR Code",ส่ง SMS ปลอมแนบลิงก์ใบแจ้งค่าเทอมในนามสถานศึกษา,สแกน QR Code หรือกดลิงก์เพื่อชำระเงิน
2,93919,2026-10-24,ข่าวปลอม,ตำรวจเปิดเพจเฟซบุ๊ก ศูนย์ให้คำปรึกษาและประสานง...,เพจปลอมมาอีกแล้ว!! โดยสวมรอยเป็นเพจหน่วยงานราช...,text,สำนักงานตำรวจแห่งชาติ,fake_authority,เพจเฟซบุ๊ก,เปิดเพจปลอมอ้างเป็นศูนย์ให้คำปรึกษาและติดตามคว...,กดลิงก์หรือเปิดเผยข้อมูลส่วนตัว
3,93911,2026-10-24,ข่าวปลอม,กรมการปกครอง เปิด TikTok ใหม่ โดยใช้ชื่อบัญชี ...,จากการตรวจสอบ พบผู้ไม่หวังดีเปิดบัญชี TikTok ใ...,text,กรมการปกครอง,fake_authority,TikTok,เปิดบัญชี TikTok ปลอมอ้างเป็นกรมการปกครอง,ส่งข้อมูลส่วนตัวหรือทำธุรกรรม
4,93905,2026-10-24,อาชญากรรมออนไลน์,ทุนการศึกษาปลอม หลอกเก็บค่าสมัคร,เหล่านักเรียน นักศึกษา ที่กำลังมองหาทุนเพื่อแบ...,text,,fake_authority,เพจเฟซบุ๊ก,สร้างเพจรับสมัครทุนการศึกษาปลอมเรียกเก็บค่าประกัน,กรอกใบสมัครและส่งเอกสารส่วนตัว
